

# Feature Extraction - SIFT (특징점 추출)

## 개요

**SIFT (Scale-Invariant Feature Transform)**는 David Lowe가 1999년에 제안한 특징점 검출 및 기술(descriptor) 알고리즘입니다. SIFT는 **스케일 불변성(scale invariance)**과 **회전 불변성(rotation invariance)**을 가지며, SfM 파이프라인의 첫 번째 단계로서 이미지에서 안정적인 특징점을 추출합니다.

### 특허 이슈

SIFT는 원래 **특허**로 보호되어 상업적 사용에 제한이 있었습니다:

| 항목 | 내용 |
|------|------|
| 특허 번호 | US 6,711,293 |
| 출원일 | 1999년 3월 6일 |
| **만료일** | **2020년 3월 6일** |

**현재 상태 (2020년 이후)**:
- ✅ 특허 만료로 **자유롭게 사용 가능**
- ✅ OpenCV 4.4.0부터 기본 모듈에 SIFT 포함 (이전에는 `opencv-contrib`의 `nonfree` 모듈)
- ✅ COLMAP, VLFeat 등에서 제한 없이 사용

> **역사적 배경**: 특허 기간 동안 SURF, ORB, BRISK 등 대안 알고리즘들이 개발되었습니다. 특히 ORB는 특허-free이면서 빠르다는 장점으로 실시간 응용에서 인기를 얻었습니다.

---

## 왜 불변성(Invariance)이 중요한가?

SfM에서는 **같은 3D 점**을 **다른 위치/각도/거리**에서 촬영한 이미지들에서 매칭해야 합니다. 불변성이 없으면 같은 물체도 다르게 인식됩니다.

### Scale Invariance (스케일 불변성)

**문제**: 같은 물체도 카메라와의 거리에 따라 크기가 달라집니다.

```
가까이 (확대)              멀리 (축소)
┌─────────────┐           ┌───┐
│   ┌───┐     │           │ ■ │  ← 같은 코너인데
│   │ ■ │     │           └───┘    크기가 완전히 다름
│   └───┘     │
└─────────────┘
```



**해결**: **Scale-Space**에서 특징점 검출
- 여러 스케일($\sigma$)의 Gaussian blur 이미지를 생성
- DoG (Difference of Gaussians)에서 extrema를 찾아 특징점의 **고유 스케일** 결정
- 검출된 스케일에서 descriptor 계산 → 크기가 달라도 같은 descriptor

### Rotation Invariance (회전 불변성)

**문제**: 카메라 각도에 따라 같은 물체도 회전되어 보입니다.

```
이미지 A                    이미지 B (30° 회전)
┌─────────────┐            ┌─────────────┐
│    ★        │            │      ★      │
│   ↑         │            │     ↗       │
│ 주방향=90°  │            │  주방향=120° │
└─────────────┘            └─────────────┘
```



**해결**: **방향 정규화** 후 descriptor 계산
1. Keypoint 주변의 gradient 히스토그램으로 **주 방향(dominant orientation)** 계산
2. Patch를 주 방향으로 **정렬한 후** descriptor 계산
3. 결과: 회전되어도 **같은 descriptor** 생성

```
정규화 전 (다름)            정규화 후 (같음)
  A: ↑                        A: ↑
  B: ↗                        B: ↑  (← 0°로 정렬)
```



### 불변성 vs 매칭 정확도 Trade-off

| 옵션 | Scale Invariant | Rotation Invariant | 용도 |
|------|-----------------|-------------------|------|
| 기본 SIFT | ✅ | ✅ | 일반 장면 |
| `upright=true` | ✅ | ❌ | 건물/도시 (중력 방향 일정) |
| `first_octave=0` | 제한적 | ✅ | 빠른 처리 |

> **참고**: 회전 불변성을 끄면 (`upright=true`) 매칭 ambiguity가 줄어들어 정확도가 향상될 수 있습니다.

---

## 이론적 배경

### 1. Scale Space와 DoG (Difference of Gaussians)

#### Gaussian Blur
이미지 $I(x, y)$에 Gaussian kernel을 적용하여 다양한 스케일의 blur 이미지를 생성:

$$
L(x, y, \sigma) = G(x, y, \sigma) * I(x, y)
$$

여기서 Gaussian kernel:

$$
G(x, y, \sigma) = \frac{1}{2\pi\sigma^2} e^{-\frac{x^2 + y^2}{2\sigma^2}}
$$

#### Difference of Gaussians (DoG)
LoG (Laplacian of Gaussian)의 효율적인 근사:

$$
D(x, y, \sigma) = L(x, y, k\sigma) - L(x, y, \sigma)
$$

$k$는 스케일 배율 (일반적으로 $k = 2^{1/s}$, $s$는 octave당 level 수)

#### Octave와 Scale Pyramid

```
Octave 0 (원본 해상도):    σ₀, k·σ₀, k²·σ₀, ...
Octave 1 (½ 해상도):       2σ₀, 2k·σ₀, 2k²·σ₀, ...
Octave 2 (¼ 해상도):       4σ₀, 4k·σ₀, 4k²·σ₀, ...
```



### 2. Keypoint Detection (특징점 검출)

#### Extrema 검출
DoG 공간에서 3×3×3 이웃에서의 local extrema를 찾음:

$$
D(x, y, \sigma) \text{ is extremum if } D > \text{all 26 neighbors} \text{ or } D < \text{all 26 neighbors}
$$

#### Sub-pixel 정제
Taylor expansion을 통한 정확한 위치 추정:

$$
D(\mathbf{x}) \approx D + \frac{\partial D^T}{\partial \mathbf{x}} \mathbf{x} + \frac{1}{2} \mathbf{x}^T \frac{\partial^2 D}{\partial \mathbf{x}^2} \mathbf{x}
$$

최적 위치:
$$
\hat{\mathbf{x}} = -\left(\frac{\partial^2 D}{\partial \mathbf{x}^2}\right)^{-1} \frac{\partial D}{\partial \mathbf{x}}
$$

### 3. Keypoint Filtering (불량 특징점 제거)

#### Low Contrast Rejection
극값에서의 함수값이 threshold보다 작으면 제거:

$$
|D(\hat{\mathbf{x}})| < \text{peak\_threshold}
$$

#### Edge Response Rejection
Hessian matrix의 eigenvalue 비율로 edge 검출:

$$
H = \begin{bmatrix} D_{xx} & D_{xy} \\ D_{xy} & D_{yy} \end{bmatrix}
$$

Edge ratio test:
$$
\frac{\text{Tr}(H)^2}{\text{Det}(H)} < \frac{(r+1)^2}{r}
$$

여기서 $r$은 edge threshold (보통 10)

### 4. Orientation Assignment (방향 할당)

Keypoint 주변의 gradient 방향 히스토그램 계산:

$$
m(x, y) = \sqrt{(L(x+1, y) - L(x-1, y))^2 + (L(x, y+1) - L(x, y-1))^2}
$$

$$
\theta(x, y) = \tan^{-1}\left(\frac{L(x, y+1) - L(x, y-1)}{L(x+1, y) - L(x-1, y)}\right)
$$

- 36-bin 히스토그램 (10° 간격)
- 가장 높은 peak의 80% 이상인 방향들도 keypoint로 생성

### 5. SIFT Descriptor (128차원 기술자)

#### 구조
- 4×4 cell grid (keypoint 중심)
- 각 cell에 8-bin 방향 히스토그램
- 총 $4 \times 4 \times 8 = 128$ 차원

#### 계산 과정
1. Keypoint 방향으로 patch 회전 (rotation invariance)
2. 16×16 pixel 영역을 4×4 cell로 분할
3. 각 cell에서 gradient 방향 히스토그램 계산
4. Gaussian 가중치 적용 (중심에서 멀수록 감소)

#### 정규화
1. L2 정규화: $\mathbf{d} \leftarrow \mathbf{d} / \|\mathbf{d}\|$
2. Clipping: $d_i \leftarrow \min(d_i, 0.2)$ (조명 변화에 강건)
3. 재정규화: $\mathbf{d} \leftarrow \mathbf{d} / \|\mathbf{d}\|$

---

## COLMAP 구현 분석

### SIFT Options 구조

In [ ]:
// sift.h - SIFT 추출 옵션

struct SiftExtractionOptions {
  // 최대 특징점 수 (큰 스케일 특징점 우선 유지)
  int max_num_features = 8192;

  // 첫 번째 octave (-1: 이미지를 2배로 업샘플링)
  // -1로 설정하면 더 세밀한 특징점 검출 가능
  int first_octave = -1;

  // Octave 수 (스케일 피라미드 레벨)
  int num_octaves = 4;

  // Octave당 스케일 level 수
  int octave_resolution = 3;

  // Peak threshold (낮을수록 더 많은 특징점)
  // octave_resolution으로 나누어 정규화
  double peak_threshold = 0.02 / octave_resolution;

  // Edge threshold (높을수록 더 많은 edge 허용)
  double edge_threshold = 10.0;

  // Affine shape 추정 (타원형 vs 원형)
  bool estimate_affine_shape = false;

  // Keypoint당 최대 방향 수
  int max_num_orientations = 2;

  // Upright features (회전 불변성 비활성화)
  bool upright = false;

  // 어두운 이미지 적응 (SiftGPU OpenGL 전용)
  bool darkness_adaptivity = false;

  // Domain-Size Pooling (향상된 descriptor)
  bool domain_size_pooling = false;
  double dsp_min_scale = 1.0 / 6.0;
  double dsp_max_scale = 3.0;
  int dsp_num_scales = 10;

  // Descriptor 정규화 방법
  // L1_ROOT: L1 정규화 후 sqrt (권장, 성능 더 좋음)
  // L2: 표준 L2 정규화
  enum class Normalization { L1_ROOT, L2 };
  Normalization normalization = Normalization::L1_ROOT;
};



**파라미터 의미:**

| 파라미터 | 기본값 | 의미 |
|---------|-------|------|
| `max_num_features` | 8192 | 이미지당 최대 특징점 수 |
| `first_octave` | -1 | 시작 octave (-1: 2x upsampling) |
| `num_octaves` | 4 | 스케일 피라미드 depth |
| `octave_resolution` | 3 | Octave당 DoG level 수 |
| `peak_threshold` | 0.0067 | 낮은 contrast 제거 threshold |
| `edge_threshold` | 10.0 | Edge 제거 threshold |

### Feature Keypoint 구조

In [ ]:
// types.h - Keypoint 데이터 구조

struct FeatureKeypoint {
  FeatureKeypoint();
  FeatureKeypoint(float x, float y);
  FeatureKeypoint(float x, float y, float scale, float orientation);
  FeatureKeypoint(float x, float y, float a11, float a12, float a21, float a22);

  // 위치 (원점: 좌상단, (0.5, 0.5)이 첫 픽셀 중심)
  float x;
  float y;

  // Affine shape (2x2 행렬로 표현)
  // 원형 특징점: a11 = a22 = scale, a12 = a21 = 0
  // 타원형 특징점: affine transformation
  float a11;  // scale * cos(orientation)
  float a12;  // scale * sin(orientation)  
  float a21;  // -scale * sin(orientation)
  float a22;  // scale * cos(orientation)

  // Shape 파라미터 계산
  float ComputeScale() const;
  float ComputeScaleX() const;
  float ComputeScaleY() const;
  float ComputeOrientation() const;
  float ComputeShear() const;

  // 리스케일
  void Rescale(float scale);
  void Rescale(float scale_x, float scale_y);
};

// Descriptor: Nx128 matrix (N개 keypoint, 128차원)
typedef Eigen::Matrix<uint8_t, Eigen::Dynamic, Eigen::Dynamic, Eigen::RowMajor>
    FeatureDescriptors;



**Affine Shape과 Scale/Orientation의 관계:**

$$
\begin{bmatrix} a_{11} & a_{12} \\ a_{21} & a_{22} \end{bmatrix} = 
s \cdot \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}
$$

여기서 $s$는 scale, $\theta$는 orientation

### CPU SIFT Extractor (VLFeat 기반)

In [ ]:
// sift.cc - VLFeat 라이브러리 사용

class SiftCPUFeatureExtractor : public FeatureExtractor {
 public:
  using VlSiftType = std::unique_ptr<VlSiftFilt, void (*)(VlSiftFilt*)>;

  explicit SiftCPUFeatureExtractor(const FeatureExtractionOptions& options)
      : options_(options), sift_(nullptr, &vl_sift_delete) {
    THROW_CHECK(options_.Check());
    // CPU 버전은 affine shape, domain size pooling 미지원
    THROW_CHECK(!options_.sift->estimate_affine_shape);
    THROW_CHECK(!options_.sift->domain_size_pooling);
  }

  bool Extract(const Bitmap& bitmap,
               FeatureKeypoints* keypoints,
               FeatureDescriptors* descriptors) {
    THROW_CHECK(bitmap.IsGrey());  // 그레이스케일 필수
    THROW_CHECK_NOTNULL(keypoints);

    // VLFeat SIFT 필터 생성/재사용
    if (sift_ == nullptr || sift_->width != bitmap.Width() ||
        sift_->height != bitmap.Height()) {
      sift_ = VlSiftType(
          vl_sift_new(bitmap.Width(),
                      bitmap.Height(),
                      options_.sift->num_octaves,
                      options_.sift->octave_resolution,
                      options_.sift->first_octave),
          &vl_sift_delete);
    }

    // Threshold 설정
    vl_sift_set_peak_thresh(sift_.get(), options_.sift->peak_threshold);
    vl_sift_set_edge_thresh(sift_.get(), options_.sift->edge_threshold);

    // 이미지 데이터를 float으로 변환 [0, 1]
    const std::vector<uint8_t>& data_uint8 = bitmap.RowMajorData();
    std::vector<float> data_float(data_uint8.size());
    for (size_t i = 0; i < data_uint8.size(); ++i) {
      data_float[i] = static_cast<float>(data_uint8[i]) / 255.0f;
    }

    // Octave별 처리
    std::vector<FeatureKeypoints> level_keypoints;
    std::vector<FeatureDescriptors> level_descriptors;
    
    // 첫 번째 octave 처리
    if (vl_sift_process_first_octave(sift_.get(), data_float.data())) {
      return true;  // 실패
    }
    
    while (true) {
      // Keypoint 검출
      vl_sift_detect(sift_.get());

      const VlSiftKeypoint* vl_keypoints = vl_sift_get_keypoints(sift_.get());
      const int num_keypoints = vl_sift_get_nkeypoints(sift_.get());

      // 각 keypoint에 대해 orientation 계산
      for (int i = 0; i < num_keypoints; ++i) {
        double angles[4];
        int num_orientations;
        
        if (options_.sift->upright) {
          // Upright: 방향 고정 (0)
          num_orientations = 1;
          angles[0] = 0.0;
        } else {
          // 방향 계산 (최대 4개의 peak)
          num_orientations = vl_sift_calc_keypoint_orientations(
              sift_.get(), angles, &vl_keypoints[i]);
        }

        // 사용할 방향 수 제한
        const int num_used = std::min(
            num_orientations, options_.sift->max_num_orientations);

        for (int o = 0; o < num_used; ++o) {
          // Keypoint 저장
          level_keypoints.back().push_back(
              FeatureKeypoint(vl_keypoints[i].x + 0.5f,  // pixel 중심
                              vl_keypoints[i].y + 0.5f,
                              vl_keypoints[i].sigma,     // scale
                              angles[o]));               // orientation

          // Descriptor 계산
          if (descriptors != nullptr) {
            FeatureDescriptorsFloat desc(1, 128);
            vl_sift_calc_keypoint_descriptor(
                sift_.get(), desc.data(), &vl_keypoints[i], angles[o]);
            
            // 정규화
            if (options_.sift->normalization == 
                SiftExtractionOptions::Normalization::L2) {
              L2NormalizeFeatureDescriptors(&desc);
            } else {  // L1_ROOT
              L1RootNormalizeFeatureDescriptors(&desc);
            }
            
            // float → uint8 변환
            level_descriptors.back().row(level_idx) =
                FeatureDescriptorsToUnsignedByte(desc);
          }
        }
      }

      // 다음 octave로 이동
      if (vl_sift_process_next_octave(sift_.get())) {
        break;  // 마지막 octave
      }
    }

    // max_num_features 제한 적용 (큰 스케일 우선)
    // ...
    
    return true;
  }

 private:
  const FeatureExtractionOptions options_;
  VlSiftType sift_;
};



### Covariant SIFT (Affine Shape 지원)

In [ ]:
// sift.cc - Covariant detector (affine invariant)

class CovariantSiftCPUFeatureExtractor : public FeatureExtractor {
 public:
  bool Extract(const Bitmap& bitmap,
               FeatureKeypoints* keypoints,
               FeatureDescriptors* descriptors) {
    // VLFeat의 Covariant Detector 사용
    std::unique_ptr<VlCovDet, void (*)(VlCovDet*)> covdet(
        vl_covdet_new(VL_COVDET_METHOD_DOG),  // DoG 방법 사용
        &vl_covdet_delete);

    // 옵션 설정
    vl_covdet_set_first_octave(covdet.get(), options_.sift->first_octave);
    vl_covdet_set_octave_resolution(covdet.get(), options_.sift->octave_resolution);
    vl_covdet_set_peak_threshold(covdet.get(), options_.sift->peak_threshold);
    vl_covdet_set_edge_threshold(covdet.get(), options_.sift->edge_threshold);

    // 이미지 데이터 변환
    // ...

    // Keypoint 검출
    vl_covdet_detect(covdet.get());

    // Affine shape 추정 (옵션)
    if (options_.sift->estimate_affine_shape) {
      vl_covdet_extract_affine_shape(covdet.get());
    }

    // Orientation 추정
    if (!options_.sift->upright) {
      vl_covdet_extract_orientations(covdet.get());
    }

    // Keypoint 추출
    const int num_features = vl_covdet_get_num_features(covdet.get());
    const VlCovDetFeature* features = vl_covdet_get_features(covdet.get());

    for (int i = 0; i < num_features; ++i) {
      // Affine frame: [A | t] 형태
      // feature.frame.a11, a12, a21, a22: affine shape
      // feature.frame.x, y: 위치
      keypoints->emplace_back(
          features[i].frame.x + 0.5f,
          features[i].frame.y + 0.5f,
          features[i].frame.a11,
          features[i].frame.a12,
          features[i].frame.a21,
          features[i].frame.a22);
    }

    // Descriptor 계산 (Domain-Size Pooling 지원)
    if (descriptors != nullptr) {
      // Domain-Size Pooling: 여러 스케일에서 descriptor 평균
      if (options_.sift->domain_size_pooling) {
        // 여러 스케일에서 descriptor 추출 후 평균
        // ...
      } else {
        // 단일 스케일 descriptor
        // ...
      }
    }

    return true;
  }
};



### Descriptor 정규화

In [ ]:
// VLFeat → UBC 포맷 변환 (순서 재배열)
FeatureDescriptors TransformVLFeatToUBCFeatureDescriptors(
    const FeatureDescriptors& vlfeat_descriptors) {
  FeatureDescriptors ubc_descriptors(vlfeat_descriptors.rows(),
                                     vlfeat_descriptors.cols());
  // VLFeat과 UBC(SiftGPU)의 bin 순서가 다름
  const std::array<int, 8> q{{0, 7, 6, 5, 4, 3, 2, 1}};
  for (int n = 0; n < vlfeat_descriptors.rows(); ++n) {
    for (int i = 0; i < 4; ++i) {
      for (int j = 0; j < 4; ++j) {
        for (int k = 0; k < 8; ++k) {
          ubc_descriptors(n, 8 * (j + 4 * i) + q[k]) =
              vlfeat_descriptors(n, 8 * (j + 4 * i) + k);
        }
      }
    }
  }
  return ubc_descriptors;
}

// L1-Root 정규화 (RootSIFT)
// L1 정규화 후 element-wise sqrt
// 성능이 L2 정규화보다 우수함 (Arandjelovic, Zisserman CVPR 2012)
void L1RootNormalizeFeatureDescriptors(FeatureDescriptorsFloat* descriptors) {
  for (int i = 0; i < descriptors->rows(); ++i) {
    const float l1_norm = descriptors->row(i).lpNorm<1>();
    if (l1_norm > 0) {
      descriptors->row(i) /= l1_norm;
      descriptors->row(i) = descriptors->row(i).cwiseSqrt();
    }
  }
}

// L2 정규화 (표준 SIFT)
void L2NormalizeFeatureDescriptors(FeatureDescriptorsFloat* descriptors) {
  for (int i = 0; i < descriptors->rows(); ++i) {
    const float l2_norm = descriptors->row(i).norm();
    if (l2_norm > 0) {
      descriptors->row(i) /= l2_norm;
    }
  }
}



### GPU SIFT (SiftGPU)

In [ ]:
// sift.cc - SiftGPU 사용 (CUDA 또는 OpenGL)

#if defined(COLMAP_GPU_ENABLED)
#include "thirdparty/SiftGPU/SiftGPU.h"

class SiftGPUFeatureExtractor : public FeatureExtractor {
 public:
  SiftGPUFeatureExtractor(const FeatureExtractionOptions& options,
                          SiftGPU* sift_gpu)
      : options_(options), sift_gpu_(sift_gpu) {}

  bool Extract(const Bitmap& bitmap,
               FeatureKeypoints* keypoints,
               FeatureDescriptors* descriptors) {
    // GPU로 이미지 업로드 및 특징점 추출
    const int code = sift_gpu_->RunSIFT(
        bitmap.Width(), bitmap.Height(),
        bitmap.RowMajorData().data(),
        GL_LUMINANCE,     // 그레이스케일
        GL_UNSIGNED_BYTE);

    if (code != 1) {
      return false;
    }

    // 결과 가져오기
    const int num_features = sift_gpu_->GetFeatureNum();
    
    // SiftGPU 포맷의 keypoint (x, y, scale, orientation)
    std::vector<SiftGPU::SiftKeypoint> sift_keypoints(num_features);
    
    // 128차원 float descriptor
    std::vector<float> sift_descriptors(128 * num_features);
    
    sift_gpu_->GetFeatureVector(
        sift_keypoints.data(), sift_descriptors.data());

    // COLMAP 포맷으로 변환
    keypoints->resize(num_features);
    for (int i = 0; i < num_features; ++i) {
      (*keypoints)[i] = FeatureKeypoint(
          sift_keypoints[i].x,
          sift_keypoints[i].y,
          sift_keypoints[i].s,  // scale
          sift_keypoints[i].o); // orientation
    }

    if (descriptors != nullptr) {
      // float → uint8 변환 (0-512 범위를 0-255로)
      // ...
    }

    return true;
  }

 private:
  const FeatureExtractionOptions options_;
  SiftGPU* sift_gpu_;
};
#endif  // COLMAP_GPU_ENABLED



---

## 이론과 코드 매핑

| 이론적 개념 | COLMAP 구현 |
|------------|-------------|
| Scale-space $L(x,y,\sigma)$ | `vl_sift_process_first_octave()` |
| DoG $D(x,y,\sigma)$ | VLFeat 내부 계산 |
| Extrema detection | `vl_sift_detect()` |
| Peak threshold | `peak_threshold` 옵션 |
| Edge rejection (Hessian) | `edge_threshold` 옵션 |
| Orientation histogram | `vl_sift_calc_keypoint_orientations()` |
| 128-D descriptor | `vl_sift_calc_keypoint_descriptor()` |
| L1-Root normalization | `L1RootNormalizeFeatureDescriptors()` |
| Affine shape | `vl_covdet_extract_affine_shape()` |

---

## 실용적 고려사항

### 1. 파라미터 튜닝

In [ ]:
// 더 많은 특징점을 원할 때
options.max_num_features = 16384;       // 기본: 8192
options.peak_threshold = 0.004;         // 낮출수록 더 많은 특징점
options.edge_threshold = 15.0;          // 높일수록 더 많은 edge 허용

// 더 빠른 추출을 원할 때
options.max_num_features = 4096;        // 줄이기
options.first_octave = 0;               // upsampling 비활성화
options.num_octaves = 3;                // 줄이기

// 더 강건한 매칭을 원할 때
options.domain_size_pooling = true;     // 향상된 descriptor
options.estimate_affine_shape = true;   // viewpoint 변화에 강건



### 2. CPU vs GPU

| 구분 | CPU (VLFeat) | GPU (SiftGPU) |
|-----|-------------|---------------|
| 속도 | 느림 | 10-100x 빠름 |
| 메모리 | 낮음 | GPU 메모리 필요 |
| 정확도 | 기준 | 약간 다름 (구현 차이) |
| 기능 | Affine, DSP 지원 | 기본 SIFT만 |
| 배치 | 어려움 | 쉬움 |

### 3. 특징점 수 vs 품질 Trade-off

```
max_num_features = 2000:  빠름, sparse reconstruction
max_num_features = 8192:  균형 (기본값)
max_num_features = 32000: 느림, dense reconstruction
```



### 4. Upright vs Rotation-Invariant

In [ ]:
// 건물, 도시 장면 (중력 방향 일정)
options.upright = true;   // 회전 불변성 비활성화, 매칭 정확도 향상

// 일반 장면
options.upright = false;  // 기본값, 회전 불변



### 5. Domain-Size Pooling

In [ ]:
// 향상된 descriptor (느리지만 성능 좋음)
options.domain_size_pooling = true;
options.dsp_min_scale = 1.0 / 6.0;  // 최소 스케일 비율
options.dsp_max_scale = 3.0;         // 최대 스케일 비율
options.dsp_num_scales = 10;         // 평균할 스케일 수



---

## Feature Extraction 파이프라인

```
입력 이미지
    ↓
[전처리] 그레이스케일 변환, float 정규화
    ↓
[Scale-Space] Gaussian pyramid 생성
    ↓
[DoG] Difference of Gaussians 계산
    ↓
[Extrema Detection] 3x3x3 local extrema 검출
    ↓
[Sub-pixel Refinement] Taylor expansion으로 정밀 위치
    ↓
[Filtering] Low contrast, edge response 제거
    ↓
[Orientation] Gradient histogram → 주 방향 할당
    ↓
[Descriptor] 4x4x8 = 128차원 descriptor 계산
    ↓
[Normalization] L1-Root 또는 L2 정규화
    ↓
출력: Keypoints [(x, y, scale, orientation)] + Descriptors [128-D]
```



---

## 참고 자료

1. Lowe, D.G. "Object Recognition from Local Scale-Invariant Features" (ICCV 1999) - 최초 발표
2. Lowe, D.G. "Distinctive Image Features from Scale-Invariant Keypoints" (IJCV 2004) - 표준 레퍼런스
3. Vedaldi, Fulkerson. "VLFeat: An Open and Portable Library of Computer Vision Algorithms" (2008)
4. Arandjelovic, Zisserman. "Three things everyone should know to improve object retrieval" (CVPR 2012) - RootSIFT
5. Dong, Soatto. "Domain-Size Pooling in Local Descriptors" (CVPR 2015)
6. [Overview | SIFT Detector](https://www.youtube.com/watch?v=KgsHoJYJ4S8)